# GPU-over-IP Colab live experiment
Runs on your Colab GPU; applications stay on Windows 10. Success = Windows Juice client sees this GPU and executes a local graphics workload. No media-upload server or remote desktop.

In [ ]:
import os,subprocess,shutil,json,time,pathlib,platform
r=lambda a: subprocess.run(a,text=True,capture_output=True,timeout=30)
print('HOST',platform.platform())
print('NVIDIA_DEVICES',list(map(str,pathlib.Path('/dev').glob('nvidia*'))))
if shutil.which('nvidia-smi'):
 p=r(['nvidia-smi','--query-gpu=name,uuid,driver_version,memory.total','--format=csv,noheader']); print('NVIDIA',p.returncode,p.stdout,p.stderr[:500])
else:print('NVIDIA_SMI_MISSING')
try:
 import torch
 print('CUDA_AVAILABLE',torch.cuda.is_available(),'GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
except Exception as e:print('CUDA_CHECK_ERROR',repr(e))

## Juice actual GPU host
This executes the real Juice agent only when the official binary, authenticated Juice token, and pool ID exist. Credentials are not embedded here. For Colab upload the official Linux Juice release obtained with your free Juice account. The host must pass Juice's GPU and network compatibility checks.

In [ ]:
import tarfile,hashlib
base=pathlib.Path('/content/gpuoverip');base.mkdir(exist_ok=True)
archive=next(pathlib.Path('/content').glob('juice-gpu-linux*.tar*'),None)
juice=shutil.which('juice')
if not juice and archive:
 target=base/'juice';target.mkdir(exist_ok=True)
 with tarfile.open(archive) as tf:
  for m in tf.getmembers():
   if m.issym() or m.islnk() or m.isdev() or not (target/m.name).resolve().is_relative_to(target): raise ValueError('Unsafe archive')
  tf.extractall(target,filter='data')
 binaries=list(target.rglob('juice'))
 if binaries:
  juice=str(binaries[0]);pathlib.Path(juice).chmod(0o755)
print('JUICE_BINARY',bool(juice))
if juice:
 p=r([juice,'check','--json']);print('JUICE_CHECK',p.returncode,p.stdout[-3000:],p.stderr[-500:])

In [ ]:
import subprocess
agent=None
pool=os.environ.get('JUICE_POOL_ID','')
token=os.environ.get('JUICE_M2M_TOKEN','')
if juice and token and pool:
 login=r([juice,'login','--token',token]);print('LOGIN_OK',login.returncode==0)
 if login.returncode==0:
  log=open(base/'juice-agent.log','w')
  agent=subprocess.Popen([juice,'agent','run',pool],stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
  time.sleep(5)
  print('JUICE_AGENT_RUNNING',agent.poll() is None)
  if agent.poll() is not None:print((base/'juice-agent.log').read_text(errors='replace')[-2000:])
else:print('JUICE_AGENT_NOT_STARTED: missing binary, token, or pool ID')

In [ ]:
if juice and agent and agent.poll() is None:
 for args in (['agent','list','--json'],['gpu','list','--json']):
  p=r([juice]+args);print('JUICE',args,p.returncode,p.stdout[-3000:],p.stderr[-400:])
else: print('NO_RUNNING_JUICE_AGENT')